In [1]:
import urllib3

print(urllib3.__file__)
print(urllib3.__version__)

c:\IMP Projects\Forest_Fire_Prediction\venv\Lib\site-packages\urllib3\__init__.py
2.7.0


In [3]:
import requests

# 1. Target Coordinates (Using Algiers, Algeria region for testing)
lat = 36.7525
lon = 3.0420

# 2. Construct clean Open-Meteo REST endpoint
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,rain",
    "timezone": "auto"
}

print(f"Contacting Open-Meteo cloud servers for coordinates ({lat}, {lon})...")

try:
    # Send direct HTTP GET request
    response = requests.get(url, params=params, timeout=10)
    
    if response.status_code == 200:
        raw_payload = response.json()
        current_metrics = raw_payload["current"]
        
        # 3. Extract the clean numerical weather variables
        live_temp = current_metrics["temperature_2m"]
        live_rh   = current_metrics["relative_humidity_2m"]
        live_ws   = current_metrics["wind_speed_10m"]
        live_rain = current_metrics["rain"]
        
        print("\n Live Weather Stream Verification ")
        print(f"Current Temperature:     {live_temp} °C")
        print(f"Current Humidity (RH):   {live_rh} %")
        print(f"Current Wind Speed (Ws): {live_ws} km/h")
        print(f"Current Hourly Rain:     {live_rain} mm")
        
    else:
        print(f"API Error! Server returned status code: {response.status_code}")
        print(f"Response content: {response.text}")

except Exception as e:
    print(f"An error occurred while connecting to the network: {str(e)}")

Contacting Open-Meteo cloud servers for coordinates (36.7525, 3.042)...

 Live Weather Stream Verification 
Current Temperature:     24.0 °C
Current Humidity (RH):   67 %
Current Wind Speed (Ws): 2.2 km/h
Current Hourly Rain:     0.0 mm


In [8]:
import numpy as np
import pandas as pd
import joblib

# 1. Load your production ML artifacts safely
try:
    model = joblib.load("../models/forest_fire_rf_model.pkl")
    scaler = joblib.load("../models/standard_scaler.pkl")
    print("✓ Production Model and Scaler loaded cleanly!")
except Exception as e:
    print(f"Error loading artifacts: {str(e)}. Make sure your paths are correct!")

# 2. Assign the live weather variables we just fetched from the API
# (Using the exact values your stream just returned)
temp = 24.0
rh   = 67.0
ws   = 2.2
rain = 0.0

print(f"\nProcessing incoming metrics... [Temp: {temp}°C, RH: {rh}%, Ws: {ws}km/h, Rain: {rain}mm]")

# 3. Complete the Preprocessing Layer: Structural Fire Index Calculations
ffmc_yesterday = 85.0
dmc_yesterday  = 24.0
dc_yesterday   = 100.0  # Adding default historical baseline for Drought Code

# Fine Fuel Moisture Code (FFMC)
ed = 0.942 * (rh**0.679) + (11.0 * np.exp((rh - 100) / 10)) + (0.18 * (100 - rh)) * (1.0 - np.exp(-0.115 * ws))
moisture_content = 147.2 * (101.0 - ffmc_yesterday) / (59.5 + ffmc_yesterday)
ko = 0.424 * (1.0 - (rh / 100)**1.7) + 0.0694 * (ws**0.5) * (1.0 - (rh / 100)**8)
kd = ko * 0.581 * np.exp(0.0365 * temp)
moisture_content_final = ed + (moisture_content - ed) * np.exp(-2.303 * kd)
calculated_ffmc = (59.5 * (147.2 - moisture_content_final)) / (147.2 + moisture_content_final)

# Initial Spread Index (ISI)
f_wind = np.exp(0.047 * ws)
f_ffmc = 91.9 * np.exp(-0.1386 * moisture_content_final) * (1.0 + (moisture_content_final**5.31) / (4.93e07))
calculated_isi = 0.208 * f_wind * f_ffmc

# Duff Moisture Code (DMC) & Drought Code (DC) Approximations
calculated_dmc = dmc_yesterday + (0.5 * temp) if temp > 0 else dmc_yesterday
calculated_dc  = dc_yesterday + (0.1 * temp) if temp > 0 else dc_yesterday

# Fire Weather Index (FWI) Approximation
calculated_fwi = 0.1 * calculated_isi * calculated_dmc

print(f"→ Computed structural indices: FFMC={calculated_ffmc:.2f}, DMC={calculated_dmc:.2f}, DC={calculated_dc:.2f}, ISI={calculated_isi:.2f}, FWI={calculated_fwi:.2f}")

# 4. Reconstruct DataFrame matching the EXACT feature alignment seen at fit time
# Ordered precisely to appease the scaler
input_data = pd.DataFrame([{
    'day': 23,
    'month': 6,
    'Temperature': temp,
    'RH': rh,
    'Ws': ws,
    'Rain': rain,
    'FFMC': calculated_ffmc,
    'DMC': calculated_dmc,
    'DC': calculated_dc,
    'ISI': calculated_isi,
    'FWI': calculated_fwi,
    'Region': 1  # Standard default region marker placeholder
}])

# Get the exact feature order from your fitted scaler
scaler_features = scaler.feature_names_in_
input_data = input_data[scaler_features]

# 5. Pass through the production scaling weights safely
input_scaled = scaler.transform(input_data)
input_scaled_df = pd.DataFrame(input_scaled, columns=scaler_features)

# 6. Generate Live Prediction Decision and Probabilities
prediction = model.predict(input_scaled_df)[0]
probabilities = model.predict_proba(input_scaled_df)[0]

print("\n___________________ LIVE RISK REPORT ___________________")
if prediction == 1:
    print(f"⚠️ STATUS: FIRE RISK DETECTED (Confidence: {probabilities[1]*100:.2f}%)")
else:
    print(f" STATUS: SAFE / NO SIGN OF FIRE RISK (Confidence: {probabilities[0]*100:.2f}%)")
print("__________________________________________________________")

✓ Production Model and Scaler loaded cleanly!

Processing incoming metrics... [Temp: 24.0°C, RH: 67.0%, Ws: 2.2km/h, Rain: 0.0mm]
→ Computed structural indices: FFMC=46.90, DMC=36.00, DC=102.40, ISI=2.04, FWI=7.35

___________________ LIVE RISK REPORT ___________________
 STATUS: SAFE / NO SIGN OF FIRE RISK (Confidence: 80.00%)
__________________________________________________________


c:\IMP Projects\Forest_Fire_Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
c:\IMP Projects\Forest_Fire_Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
